
# 40 因子数据库：PIT、分区、版本与查询

> **本章要解决的问题**：因子值算出来之后存在哪里？为什么“昨天看到的报表，今天数字对不上”？怎么在几千万行里快速取出“某一天、某因子的全市场截面”？数据改了一版，怎么知道哪些值被改过？

## 本章导读

第38章讲了流水线怎么“跑”，这一章讲流水线的**仓库**怎么建：因子值最终要落进数据库，才能被报告、组合、回测反复查询。一个能用的因子库要同时满足四件事：

1. **存得下**：多因子 × 多资产 × 多日期，行数增长很快；
2. **查得快**：最常见的查询是“某日截面”和“某因子时间序列”；
3. **看得清“当时”**：回测只能用当时可见的数据（PIT）；
4. **追得到版本**：数据修订后，能说清哪一版改了什么。

这四件事分别对应：表结构、分区与索引、bitemporal（双时间轴）、版本 diff。

### 学习目标

- 给定因子清单与数据源，能设计包含 factor、snapshot、version、universe、factor_value 五张表的最小表结构，并说清每张表的主键。
- 给定一条数据修订记录，能区分“有效时间”与“知识时间”，并写出 as-of 查询。
- 给定“按日取截面”的查询，能用物理分区与索引把全表扫描变成索引查找，并读懂执行计划。
- 给定两个时点的可见数据，能算出新增、删除、修改三类版本差异。
- 给定一张因子值表，能跑重复主键、覆盖率、异常值、universe 一致性四类质量检查。
- 给定“昨天报表和今天对不上”的故障，能定位问题出在快照、知识时间还是版本。

### 先修

第19章的库存修订与“当时可见的数据”；第38章的快照与报告 manifest；SQL 基础（SELECT、JOIN、GROUP BY）；Python 标准库 `sqlite3`。

## 在量化研究中的位置

- **谁用**：数据工程、因子研究员、回测平台、风控与合规。
- **输入**：清洗后的因子面板、批次元数据、universe 定义、修订记录。
- **输出**：可按 as-of 查询的因子库、版本差异、质量报告。
- **在流程的哪一步**：在计算之后、报告与回测之前，是所有下游的共同数据源。
- **失败模式**：用最新数据回测（偷看未来）、只覆盖不追加（历史被改写）、没有索引（查询扫全表）、版本对不上（报表无法解释）。
- **最小工作流**：建表 → 批次装载 → 建索引/分区 → as-of 查询 → 版本 diff → 质量检查。

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="40 因子数据库：PIT、分区、版本与查询" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>40 因子数据库：PIT、分区、版本与查询｜知识图</title>
<desc>本章主线：bitemporal 表管住“何时可见”，as-of 查询与版本 diff 让每一个数都能复查。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a40" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao40" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag40" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">40</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">因子数据库：PIT、分区、版本与查询</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：bitemporal 表管住“何时可见”，as-of 查询与版本 diff 让每一个数都能复查。</text>
<rect x="40" y="88" width="215" height="108" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.4" rx="14"/>
<rect x="40" y="88" width="215" height="26" fill="#1B3A5C" rx="14"/>
<rect x="40" y="101" width="215" height="13" fill="#1B3A5C"/>
<text x="52" y="106" font-size="9.8" fill="#FFFFFF" font-weight="bold">factor 因子表</text>
<text x="52" y="132" font-size="8.6" fill="#E07B39">factor_id (PK)</text>
<text x="52" y="150" font-size="8.6" fill="#1B3A5C">name / unit</text>
<text x="52" y="168" font-size="8.6" fill="#1B3A5C">direction</text>
<rect x="545" y="88" width="215" height="108" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.4" rx="14"/>
<rect x="545" y="88" width="215" height="26" fill="#1B3A5C" rx="14"/>
<rect x="545" y="101" width="215" height="13" fill="#1B3A5C"/>
<text x="557" y="106" font-size="9.8" fill="#FFFFFF" font-weight="bold">snapshot 快照表</text>
<text x="557" y="132" font-size="8.6" fill="#E07B39">snapshot_id (PK)</text>
<text x="557" y="150" font-size="8.6" fill="#1B3A5C">as_of_time</text>
<text x="557" y="168" font-size="8.6" fill="#1B3A5C">version / hash</text>
<rect x="278" y="236" width="262" height="130" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.4" rx="14"/>
<rect x="278" y="236" width="262" height="26" fill="#1B3A5C" rx="14"/>
<rect x="278" y="249" width="262" height="13" fill="#1B3A5C"/>
<text x="290" y="254" font-size="9.8" fill="#FFFFFF" font-weight="bold">value 值表</text>
<text x="290" y="280" font-size="8.6" fill="#E07B39">factor_id (FK)</text>
<text x="290" y="298" font-size="8.6" fill="#1B3A5C">trade_date</text>
<text x="290" y="316" font-size="8.6" fill="#1B3A5C">value</text>
<text x="290" y="334" font-size="8.6" fill="#E07B39">snapshot_id (FK)</text>
<text x="290" y="352" font-size="8.6" fill="#6E7B8B">PARTITION BY trade_date</text>
<rect x="40" y="262" width="200" height="92" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.4" rx="14"/>
<rect x="40" y="262" width="200" height="26" fill="#1B3A5C" rx="14"/>
<rect x="40" y="275" width="200" height="13" fill="#1B3A5C"/>
<text x="52" y="280" font-size="9.8" fill="#FFFFFF" font-weight="bold">universe 股票池</text>
<text x="52" y="306" font-size="8.6" fill="#E07B39">universe_id (PK)</text>
<text x="52" y="324" font-size="8.6" fill="#1B3A5C">date / members</text>
<path d="M 255 142 L 278 301" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 4" marker-end="url(#ag40)"/>
<path d="M 545 142 L 540 301" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 4" marker-end="url(#ag40)"/>
<path d="M 240 308 L 278 301" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 4" marker-end="url(#ag40)"/>
<rect x="241.4" y="237" width="21.1" height="14" fill="#FFFFFF"/>
<text x="252" y="246" font-size="10" fill="#6E7B8B" text-anchor="middle">1:N</text>
<rect x="529.5" y="239" width="21.1" height="14" fill="#FFFFFF"/>
<text x="540" y="248" font-size="10" fill="#6E7B8B" text-anchor="middle">1:N</text>
<rect x="229.4" y="313" width="21.1" height="14" fill="#FFFFFF"/>
<text x="240" y="322" font-size="10" fill="#6E7B8B" text-anchor="middle">1:N</text>
<text x="40" y="386" font-size="9.2" fill="#6E7B8B">as-of 查询：取 as_of_time ≤ 决策时刻 的最新版本；diff(v1, v2) 看修订影响</text>
<text x="40" y="400" font-size="9.2" fill="#1B3A5C">bitemporal：一条值同时存“发生时间”和“可见时间”</text>
</svg>

> **图**：本章主线：bitemporal 表管住“何时可见”，as-of 查询与版本 diff 让每一个数都能复查。


In [1]:

import sqlite3

import numpy as np

print("sqlite3", sqlite3.sqlite_version, "| numpy", np.__version__)


sqlite3 3.51.2 | numpy 2.4.4



## 1. 为什么因子需要数据库

**场景**：一开始因子存在 CSV 里。一年 250 个交易日、3000 只股票、20 个因子，就是 1500 万行。用 pandas 读一遍要几分钟；想做“每个调仓日取最近一期财务因子”，要在文件里反复扫；某天发现财务数据被追溯调整，却说不清哪几天的报表用了旧值。

文件系统有四个绕不过去的问题：

1. **没有约束**：同一行可以写两遍，也可以写一半；
2. **没有索引**：按日期或因子查都要全量扫描；
3. **没有事务**：写到一半崩溃，文件半新半旧；
4. **没有版本**：覆盖之后回不去。

先估算一下规模：3000 只资产 × 250 个交易日 × 20 个因子 = 1500 万行/年；再加 5 年历史、因子扩到 100 个，就是 3.75 亿行。这个体量下，“能存”不是问题，“按对的方式查”才是问题。所以设计表之前，先写下三条最频繁的查询：某日全市场截面、某因子某资产时间序列、某时点的可见版本。表结构、分区、索引都围着这三条查询转。

数据库把这四件事变成“主键 + 索引 + 事务 + 多版本行”。但要记住：**数据库不会自动解决 PIT**，时间语义仍然要你自己设计——这正是下一节的主键里那一列 `knowledge_date` 的用途。

---

## 2. 表结构：五张表各管什么

### 2.1 五张表

| 表 | 一行是什么 | 主键 |
|---|---|---|
| `factor` | 一个因子的元数据（名称、方向、频率） | `factor_id` |
| `snapshot` | 一次批量装载（批次日、数据 hash、代码版本） | `snapshot_id` |
| `version` | 因子代码/参数的一次发布 | `version_id` |
| `universe` | 某日属于某指数的成分 | `(universe_id, as_of_date, asset)` |
| `factor_value` | 一个因子、一只资产、一个交易日的值 | `(factor_id, asset, trade_date, knowledge_date)` |

五张表的分工可以这样记：`factor` 回答“这是什么因子”，`snapshot` 回答“这批数据哪来的”，`version` 回答“用哪版代码算的”，`universe` 回答“那天哪些资产算数”，`factor_value` 才是真正的数值。很多团队把所有信息塞进一张宽表，结果是每行重复存因子名、方向、版本；一旦改名或改方向，要么全表更新，要么出现两种口径并存。

### 2.2 主键的讲究

`factor_value` 的主键里多了 `knowledge_date`：同一组 `(因子, 资产, 交易日)` 允许存在多条记录，各自代表“从哪一天开始可见的值”。没有这一列，修订就只能覆盖旧值，历史版本就永远丢了。

`universe` 的主键里也有 `as_of_date`：成分股会调整，2024 年 1 月属于沪深 300 的股票，3 月可能已经被调出。回测时必须按历史成分，否则又是偷看未来。

另外两列也值得注意：`snapshot_id` 让每一行都能追溯到批次，`value` 用 REAL 存标准化后的数值。如果因子原始值量纲差异大，也可以存两个字段（原始值 + 标准化值），但表会变宽、写入变慢，按需取舍。

### 2.3 代码：建表 + 装载双时间轴数据

下面建五张表，并装载 3 个因子 × 4 只资产 × 6 个交易日 × 3 个知识日共 168 行数据。其中 2024-02-20 批次修订了一个值，2024-03-20 批次又修订了两个值——后面用它演示 as-of 和 diff。



In [2]:

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.executescript("""
CREATE TABLE factor (
    factor_id   TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    direction   INTEGER NOT NULL,
    freq        TEXT NOT NULL,
    created_at  TEXT NOT NULL
);
CREATE TABLE snapshot (
    snapshot_id  TEXT PRIMARY KEY,
    batch_date   TEXT NOT NULL,
    data_hash    TEXT NOT NULL,
    code_version TEXT NOT NULL,
    status       TEXT NOT NULL
);
CREATE TABLE version (
    version_id  TEXT PRIMARY KEY,
    factor_id   TEXT NOT NULL,
    code_hash   TEXT NOT NULL,
    params_hash TEXT NOT NULL,
    created_at  TEXT NOT NULL,
    status      TEXT NOT NULL
);
CREATE TABLE universe (
    universe_id TEXT NOT NULL,
    as_of_date  TEXT NOT NULL,
    asset       TEXT NOT NULL,
    PRIMARY KEY (universe_id, as_of_date, asset)
);
CREATE TABLE factor_value (
    factor_id      TEXT NOT NULL,
    asset          TEXT NOT NULL,
    trade_date     TEXT NOT NULL,
    knowledge_date TEXT NOT NULL,
    snapshot_id    TEXT NOT NULL,
    value          REAL NOT NULL,
    PRIMARY KEY (factor_id, asset, trade_date, knowledge_date)
);
""")

factors = [("rev20", "20 日反转", -1), ("mom60", "60 日动量", 1), ("lowvol", "低波", 1)]
assets = ["A001", "A002", "A003", "A004"]
trade_dates = ["2024-01-02", "2024-01-15", "2024-02-01", "2024-02-15", "2024-03-01", "2024-03-15"]

cur.executemany("INSERT INTO factor VALUES (?,?,?,?,?)",
                [(fid, name, d, "D", "2024-01-01") for fid, name, d in factors])
cur.executemany("INSERT INTO snapshot VALUES (?,?,?,?,?)", [
    ("snap-01", "2024-01-20", "h1a2c3", "git:7f3a2c1", "published"),
    ("snap-02", "2024-02-20", "h4d5e6", "git:7f3a2c1", "published"),
    ("snap-03", "2024-03-20", "h7f8a9", "git:9b1c4d2", "published"),
])
cur.executemany("INSERT INTO version VALUES (?,?,?,?,?,?)", [
    ("ver-01", "rev20", "code-a", "p-a", "2024-01-20", "published"),
    ("ver-02", "rev20", "code-b", "p-a", "2024-02-20", "published"),
    ("ver-03", "lowvol", "code-a", "p-b", "2024-03-20", "published"),
])
cur.executemany("INSERT INTO universe VALUES (?,?,?)",
                [("u_csi300", d, a) for d in trade_dates for a in assets])
conn.commit()

rng = np.random.default_rng(4001)
base = {}
for fid, _, _ in factors:
    for a in assets:
        for d in trade_dates:
            base[(fid, a, d)] = round(float(rng.normal(0, 1)), 4)

override_v2 = {("rev20", "A002", "2024-01-15"): 0.05}
override_v3 = {("mom60", "A003", "2024-01-02"): -0.07, ("lowvol", "A001", "2024-01-15"): 0.10}


def value_at(key, level):
    val = base[key]
    if level >= 2 and key in override_v2:
        val += override_v2[key]
    if level >= 3 and key in override_v3:
        val += override_v3[key]
    return round(val, 4)


rows = []
for fid, _, _ in factors:
    for a in assets:
        for d in trade_dates:
            key = (fid, a, d)
            if d < "2024-02-01":
                rows.append((fid, a, d, "2024-01-20", "snap-01", value_at(key, 1)))
            rows.append((fid, a, d, "2024-02-20", "snap-02", value_at(key, 2)))
            rows.append((fid, a, d, "2024-03-20", "snap-03", value_at(key, 3)))
cur.executemany("INSERT INTO factor_value VALUES (?,?,?,?,?,?)", rows)
conn.commit()

print("因子表", cur.execute("SELECT factor_id, direction FROM factor").fetchall())
print("快照表", cur.execute("SELECT snapshot_id, batch_date FROM snapshot").fetchall())
print("值表行数", cur.execute("SELECT COUNT(*) FROM factor_value").fetchone()[0])
print("每快照行数", cur.execute(
    "SELECT snapshot_id, COUNT(*) FROM factor_value GROUP BY snapshot_id ORDER BY snapshot_id").fetchall())


因子表 [('rev20', -1), ('mom60', 1), ('lowvol', 1)]
快照表 [('snap-01', '2024-01-20'), ('snap-02', '2024-02-20'), ('snap-03', '2024-03-20')]
值表行数 168
每快照行数 [('snap-01', 24), ('snap-02', 72), ('snap-03', 72)]



**读数**：值表 168 行；三个快照分别是 24、72、72 行——第一个快照只覆盖 1 月，后两个覆盖全季。`snapshot` 表记录了每次装载的 hash 和代码版本，`version` 表记录了因子代码/参数的发布；两者合起来才能回答“这批数据是谁、用什么代码、什么时候写进来的”。

**常见错误**：

1. 只用一张宽表，不留 `knowledge_date`——修订只能覆盖，历史查不回来。
2. universe 不按日期存——回测用了今天的成分股，偷看未来。
3. 快照表只记日期不记 hash——数据换了也看不出来。

---

## 3. bitemporal 与 as-of 查询

### 3.1 两条时间轴

bitemporal 的意思是：每条事实都有两个时间。

- **有效时间（valid time / `trade_date`）**：这条数据描述的是哪一天的事实；
- **知识时间（knowledge time / `knowledge_date`）**：这条数据从哪一天开始被系统知道。

举例：某股票 2024-01-15 的反转因子，最初在 2024-01-20 入库，值 1.2562；后来在 2024-03-20 被修订为 1.3062。同一条“1 月 15 日的事实”因此有两行，区别只在 `knowledge_date`。

注意这里没有“更新”操作：修订不是 `UPDATE`，而是 `INSERT` 一条知识日更晚的新行。这样任何历史时点的查询都能还原，代价是表会变长——这也是为什么分区、索引和定期归档必须一起设计。

### 3.2 as-of 查询的语义

as-of 查询回答的是：“站在某一天，我能看到什么？”标准写法分两步：

1. 对每个 `(factor_id, asset, trade_date)`，取 `knowledge_date <= as_of` 中最大的那条；
2. 再按 `trade_date <= as_of` 过滤（回测日之后的数据不可见）。

第38章和第19章讲的是同一件事：**回测必须站在过去的那一天看数据**。区别只在实现：第19章用文件和小表演示，这一章做成数据库查询。

### 3.3 代码：as-of 查询



In [3]:

ASOF_SQL = """
WITH latest AS (
    SELECT factor_id, asset, trade_date, MAX(knowledge_date) AS kd
    FROM factor_value
    WHERE knowledge_date <= ? AND trade_date <= ?
    GROUP BY factor_id, asset, trade_date
)
SELECT v.factor_id, v.asset, v.trade_date, v.value, v.knowledge_date
FROM factor_value AS v
JOIN latest AS l
  ON v.factor_id = l.factor_id AND v.asset = l.asset
 AND v.trade_date = l.trade_date AND v.knowledge_date = l.kd
WHERE (? IS NULL OR v.factor_id = ?)
ORDER BY v.factor_id, v.trade_date, v.asset
"""


def asof_values(as_of, factor_id=None):
    return cur.execute(ASOF_SQL, (as_of, as_of, factor_id, factor_id)).fetchall()


for as_of in ["2024-02-10", "2024-03-25"]:
    got = asof_values(as_of)
    print(f"as-of {as_of} 可见行数 {len(got)}")
print("as-of 2024-02-10 的 rev20 前 6 行:")
for row in asof_values("2024-02-10", "rev20")[:6]:
    print("   ", row)


as-of 2024-02-10 可见行数 24
as-of 2024-03-25 可见行数 72
as-of 2024-02-10 的 rev20 前 6 行:
    ('rev20', 'A001', '2024-01-02', 0.6586, '2024-01-20')
    ('rev20', 'A002', '2024-01-02', -0.1254, '2024-01-20')
    ('rev20', 'A003', '2024-01-02', -0.9257, '2024-01-20')
    ('rev20', 'A004', '2024-01-02', -0.8993, '2024-01-20')
    ('rev20', 'A001', '2024-01-15', -0.3437, '2024-01-20')
    ('rev20', 'A002', '2024-01-15', 1.2562, '2024-01-20')



**读数**：as-of 2024-02-10 只能看到 24 行（只有 1 月的两个交易日，且都是最早的知识日 2024-01-20）；as-of 2024-03-25 能看到 72 行（六个交易日，取各自最新的知识日）。同一份表，换一个 as-of 日期，结果完全不同——这就是 PIT 的价值。

**常见错误**：

1. 直接 `SELECT *` 取最新值回测——看到未来。
2. as-of 只过滤 `knowledge_date` 忘了过滤 `trade_date`，或者反过来。
3. 用 `MAX(value)` 之类聚合代替“取最新一行”，把修订和取值混为一谈。

---



## 4. 分区与索引：让查询变快

### 4.1 为什么要分区

因子表会随“因子数 × 资产数 × 交易日”增长，一年就可能上千万行。但查询模式高度集中：要么“某一天的截面”，要么“某因子某资产的时间序列”。按时间把大表切成小表（**物理分区**），查询时只扫相关分区，这叫分区裁剪（partition pruning）。

分区键的选择取决于查询：按交易日取截面多，就按月或按年分区；按因子查多，就按因子分区；两者都多，可以一级按年、二级按因子。分区的代价是“跨分区查询要 UNION”，分区太多也会让查询计划变复杂；一般按月分区、保留五到十年是常见折中。

### 4.2 索引：给常用查询开一条路

索引是“按某个键排好序的目录”。给 `trade_date` 建索引，按日期取截面就从“扫全表”变成“直接跳到目标行”；给 `(factor_id, trade_date)` 建联合索引，因子时间序列也能走索引。代价是写入变慢、占空间——**索引不是越多越好，要按查询模式建**。

判断一个索引值不值，看“选择性”：`trade_date` 一天只有几千行，选择性高；`direction` 只有 ±1，选择性极低。联合索引的列顺序也有讲究：等值条件在前、范围条件在后，才能同时用上两段。

### 4.3 代码：分区裁剪 + 执行计划对比

下面按月份把值表复制成三个物理分区，写一个最简单的查询路由，再对比建索引前后的执行计划。



In [4]:

PARTITIONS = {"2024-01": "value_202401", "2024-02": "value_202402", "2024-03": "value_202403"}
for month, table in PARTITIONS.items():
    cur.execute(f"DROP TABLE IF EXISTS {table}")
    cur.execute(f"CREATE TABLE {table} AS SELECT * FROM factor_value WHERE trade_date LIKE '{month}%'")
    cnt = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"物理分区 {table} 行数 {cnt}")


def query_range(start, end):
    hits = [t for m, t in sorted(PARTITIONS.items()) if start[:7] <= m <= end[:7]]
    sql = " UNION ALL ".join(
        f"SELECT factor_id, asset, trade_date, value FROM {t} WHERE trade_date BETWEEN ? AND ?"
        for t in hits
    )
    params = [p for _ in hits for p in (start, end)]
    return hits, cur.execute(sql, params).fetchall()


hits, got = query_range("2024-02-01", "2024-02-28")
print("查 2024-02 命中分区", hits, "行数", len(got))
hits, got = query_range("2024-01-01", "2024-03-31")
print("查全季命中分区数", len(hits), "行数", len(got))

QUERY_DATE = "SELECT COUNT(*) FROM factor_value WHERE trade_date = '2024-02-01'"
before = [r[-1] for r in cur.execute("EXPLAIN QUERY PLAN " + QUERY_DATE).fetchall()]
print("建索引前的计划", before)
cur.execute("CREATE INDEX idx_fv_trade_date ON factor_value(trade_date)")
cur.execute("CREATE INDEX idx_fv_factor_date ON factor_value(factor_id, trade_date)")
after = [r[-1] for r in cur.execute("EXPLAIN QUERY PLAN " + QUERY_DATE).fetchall()]
print("建索引后的计划", after)
PLAN_FACTOR = ("EXPLAIN QUERY PLAN SELECT COUNT(*) FROM factor_value "
               "WHERE factor_id = 'rev20' AND trade_date BETWEEN '2024-01-01' AND '2024-03-31'")
print("按因子取时间序列的计划", [r[-1] for r in cur.execute(PLAN_FACTOR).fetchall()])


物理分区 value_202401 行数 72
物理分区 value_202402 行数 48
物理分区 value_202403 行数 48
查 2024-02 命中分区 ['value_202402'] 行数 48
查全季命中分区数 3 行数 168
建索引前的计划 ['SCAN factor_value USING COVERING INDEX sqlite_autoindex_factor_value_1']
建索引后的计划 ['SEARCH factor_value USING COVERING INDEX idx_fv_trade_date (trade_date=?)']
按因子取时间序列的计划 ['SEARCH factor_value USING COVERING INDEX idx_fv_factor_date (factor_id=? AND trade_date>? AND trade_date<?)']



**读数**：三个物理分区分别是 72、48、48 行；查 2024-02 只命中 `value_202402` 一个分区、48 行；查全季命中 3 个分区、168 行。执行计划从 `SCAN`（扫覆盖索引）变成 `SEARCH ... USING INDEX idx_fv_trade_date (trade_date=?)`；因子时间序列的查询则走 `idx_fv_factor_date`，`factor_id` 和 `trade_date` 两个条件都用上了。

**常见错误**：

1. 分区键和查询条件不一致——按资产分区却总按日期查，分区裁剪等于没做。
2. 索引建在低选择性的列上（比如方向只有 ±1）——几乎没用还拖慢写入。
3. 只测“数据能查出来”，不测“查询计划是什么”。

---

## 5. 版本 diff：改了什么

### 5.1 三类差异

比较两个时点的可见数据，差异只有三类：

- **新增**：新时点有、旧时点没有（新交易日、新资产、新因子）；
- **删除**：旧时点有、新时点没有（少见，通常是 bug）；
- **修改**：两边都有，但值不同（修订）。

diff 必须按业务键 `(factor_id, asset, trade_date)` 做，而不是按行比——否则知识日变化会被误判成“修改”。这和第38章的“按内容判断”是一个思路。

### 5.2 代码：两个 as-of 时点的 diff



In [5]:

def asof_map(as_of):
    return {(f, a, d): (v, k) for f, a, d, v, k in asof_values(as_of)}


m_old = asof_map("2024-02-10")
m_new = asof_map("2024-03-25")
added = sorted(set(m_new) - set(m_old))
removed = sorted(set(m_old) - set(m_new))
changed = sorted(k for k in set(m_old) & set(m_new) if abs(m_old[k][0] - m_new[k][0]) > 1e-12)
print(f"旧 as-of 键数 {len(m_old)}，新 as-of 键数 {len(m_new)}")
print(f"新增 {len(added)}，删除 {len(removed)}，修改 {len(changed)}")
print("修改明细（旧值 -> 新值，知识日）:")
for key in changed:
    print("   ", key, m_old[key][0], "->", m_new[key][0], "| 知识日", m_old[key][1], "->", m_new[key][1])


旧 as-of 键数 24，新 as-of 键数 72
新增 48，删除 0，修改 3
修改明细（旧值 -> 新值，知识日）:
    ('lowvol', 'A001', '2024-01-15') -2.2867 -> -2.1867 | 知识日 2024-01-20 -> 2024-03-20
    ('mom60', 'A003', '2024-01-02') 0.3095 -> 0.2395 | 知识日 2024-01-20 -> 2024-03-20
    ('rev20', 'A002', '2024-01-15') 1.2562 -> 1.3062 | 知识日 2024-01-20 -> 2024-03-20



**读数**：旧 as-of 24 个键、新 as-of 72 个键；新增 48、删除 0、修改 3。三条修改都能精确到“旧值 → 新值”和“知识日 2024-01-20 → 2024-03-20”。这份 diff 就是“昨天报表和今天对不上”的答案：不是程序错了，是两个时点看到的数据版本不同。

diff 除了排查故障，还可以当“数据回归测试”：每次装载后自动跑一次 diff，修改条数超过阈值就告警。修改本身不一定是错——财务数据追溯调整是常态——但**修改必须可解释**：哪条规则、哪份公告、哪个批次导致的。

**常见错误**：

1. 直接对两张全量快照做行级 diff——知识日一变，整行都算“变了”。
2. 只统计修改条数，不保存明细——事后无法解释哪个值变了。
3. diff 结果不落库——每次排查都重算一遍。

---

## 6. 质量检查：四类红灯

### 6.1 四类检查

| 检查 | 问题 | 判据 |
|---|---|---|
| 重复主键 | 同一键写了两行 | 分组计数 > 1 |
| 覆盖率 | 某些日期/资产缺行 | 每期行数 = 因子数 × 资产数 |
| 异常值 | 极端值污染截面 | 按因子算 z，`|z| > 3` 计数 |
| universe 一致性 | 因子值不在成分内 | `LEFT JOIN universe` 后为空 |

再加一项“跨版本一致性”：v2 与 v3 的取值相关系数应接近 1、平均绝对变动应很小；如果突然变大，说明上游数据出了问题。质量检查要按**每一期截面**做，只看总数会漏掉“某一天缺了一半”。

质量检查最好做成“门禁”：不通过就不允许发布快照，而不是发完再补报告。检查频率可以是每次装载全量跑、每日增量跑；结果要落库，才能和上一批对比趋势。

### 6.2 代码：五类检查一次跑完



In [6]:

dup = cur.execute("""
SELECT COUNT(*) FROM (
    SELECT factor_id, asset, trade_date, knowledge_date, COUNT(*) AS c
    FROM factor_value
    GROUP BY factor_id, asset, trade_date, knowledge_date
    HAVING c > 1
)
""").fetchone()[0]
print("重复主键组数", dup)

latest_rows = asof_values("2024-03-25")
per_date = {}
for fid, a, d, v, k in latest_rows:
    per_date[d] = per_date.get(d, 0) + 1
print("各交易日截面行数", dict(sorted(per_date.items())))
print("理论值", len(factors) * len(assets))

missing = cur.execute("""
SELECT COUNT(*) FROM factor_value AS v
LEFT JOIN universe AS u
  ON u.universe_id = 'u_csi300' AND u.as_of_date = v.trade_date AND u.asset = v.asset
WHERE u.asset IS NULL
""").fetchone()[0]
print("不在 universe 中的因子值行数", missing)

print("按因子统计（最新 as-of）")
for fid, name, direction in factors:
    vals = np.array([v for f, a, d, v, k in latest_rows if f == fid])
    z = (vals - vals.mean()) / vals.std(ddof=1)
    print(f"   {fid:7s} 行数 {len(vals):3d} 均值 {vals.mean(): .4f} 标准差 {vals.std(ddof=1):.4f}"
          f" 极值 [{vals.min(): .3f}, {vals.max(): .3f}] |z|>3 个数 {(np.abs(z) > 3).sum()}")

v2 = {(f, a, d): v for f, a, d, v, k in asof_values("2024-02-25")}
v3 = {(f, a, d): v for f, a, d, v, k in asof_values("2024-03-25")}
keys = sorted(set(v2) & set(v3))
arr2 = np.array([v2[k] for k in keys])
arr3 = np.array([v3[k] for k in keys])
print("v2 与 v3 取值相关", round(float(np.corrcoef(arr2, arr3)[0, 1]), 4),
      "| 平均绝对变动", round(float(np.abs(arr3 - arr2).mean()), 5))
print("最新知识日", cur.execute("SELECT MAX(knowledge_date) FROM factor_value").fetchone()[0])
print("版本表", cur.execute("SELECT version_id, factor_id, status FROM version").fetchall())


重复主键组数 0
各交易日截面行数 {'2024-01-02': 12, '2024-01-15': 12, '2024-02-01': 12, '2024-02-15': 12, '2024-03-01': 12, '2024-03-15': 12}
理论值 12
不在 universe 中的因子值行数 0
按因子统计（最新 as-of）
   rev20   行数  24 均值  0.3211 标准差 1.1097 极值 [-1.083,  3.046] |z|>3 个数 0
   mom60   行数  24 均值  0.2367 标准差 0.9712 极值 [-1.374,  2.159] |z|>3 个数 0
   lowvol  行数  24 均值 -0.0937 标准差 0.9747 极值 [-2.469,  1.386] |z|>3 个数 0
v2 与 v3 取值相关 0.9999 | 平均绝对变动 0.00354
最新知识日 2024-03-20
版本表 [('ver-01', 'rev20', 'published'), ('ver-02', 'rev20', 'published'), ('ver-03', 'lowvol', 'published')]



**读数**：重复主键 0 组；每个交易日截面 12 行，正好等于 3 因子 × 4 资产；不在 universe 的因子值 0 行；三个因子的 `|z|>3` 个数都是 0，极值在合理范围；v2 与 v3 取值相关 0.9999，平均绝对变动 0.00354（就是那三条修订造成的）；最新知识日 2024-03-20。

**常见错误**：

1. 只查总行数，不查每期截面——少一天的数据看不出来。
2. 质量检查结果只打印不落库——下一次无法对比。
3. 把“没有异常值”当成数据质量好——缺失和覆盖才是更常见的坑。

---

## 本章小结

### 一张表

| 能力 | 手段 | 验证 |
|---|---|---|
| 存得下 | 五张表 + 主键 | 行数、覆盖率 |
| 查得快 | 物理分区 + 索引 | 执行计划 |
| 看得清当时 | bitemporal + as-of | 不同 as-of 的行数 |
| 追得到版本 | snapshot/version + diff | 新增/删除/修改 |
| 用得住 | 质量检查 | 重复/覆盖/异常/universe |

### 八句话

1. 数据库不自动解决 PIT。
2. 值表主键必须包含 `knowledge_date`。
3. universe 要按日期存，回测用历史成分。
4. as-of 查询同时过滤有效时间和知识时间。
5. 分区让查询只扫相关月份。
6. 索引按查询模式建，不是越多越好。
7. diff 按业务键比，不按行比。
8. 质量检查要落到每一期截面。

---

## 本章习题

**1.** factor、snapshot、version、universe、factor_value 五张表分别存什么？

**2.** 有效时间和知识时间有什么区别？各对应哪个字段？

**3.** as-of 查询要同时满足哪两个条件？只满足一个会怎样？

**4.** 为什么 `factor_value` 的主键要包含 `knowledge_date`？去掉它会丢什么？

**5.** 分区裁剪是什么？分区键和查询条件不一致会怎样？

**6.** 索引的收益和代价分别是什么？为什么不能给所有列都建索引？

**7.** 版本 diff 的三类差异是什么？为什么不能按行直接比？

**8.** 四类质量检查分别查什么？为什么“只看总行数”不够？

<details><summary>参考答案</summary>

**1.** factor 存因子元数据；snapshot 存每次批量装载的批次日/hash/代码版本；version 存因子代码和参数的发布记录；universe 存某日指数成分；factor_value 存因子值。

**2.** 有效时间是数据描述的哪一天（`trade_date`），知识时间是系统从哪天开始知道它（`knowledge_date`）；两者构成 bitemporal 的两条轴。

**3.** `knowledge_date <= as_of` 且 `trade_date <= as_of`；只过滤知识时间会漏掉“回测日之后才发生的事实”的排除，只过滤有效时间则会用到当时还看不到的修订。

**4.** 同一业务键可能存在多个修订版本；不包含知识日就只剩唯一一行，修订必须覆盖旧值，历史版本丢失，as-of 查询无法还原当时看到的数据。

**5.** 按时间分区后，查询只扫落在时间范围内的分区；分区键与查询条件不一致时，所有分区都要扫，裁剪失效。

**6.** 收益是查询从全表扫描变成索引查找；代价是写入变慢、占空间、维护成本；低选择性或从不被查询的列建索引几乎无收益。

**7.** 新增、删除、修改；按行比会把知识日变化误判成修改，必须按 `(factor_id, asset, trade_date)` 业务键比较，再判断值是否变化。

**8.** 重复主键、覆盖率、异常值、universe 一致性；总行数相同不代表每期都完整，可能某天缺行、另一天多行，正负抵消。

</details>

---

## 参考资料

- Kleppmann, M. *Designing Data-Intensive Applications*，第 2–3 章（数据模型、存储与索引）。
- Snodgrass, R. *Developing Time-Oriented Database Applications in SQL*（bitemporal 建模）。
- SQLite 文档：<https://www.sqlite.org/lang_with.html>（WITH 子查询）、<https://www.sqlite.org/optoverview.html>（查询计划）。
- 第19章（库存修订与当时可见的数据）；第38章（快照与报告 manifest）。
